# Nutrition, Physical Activity & Obesity — BRFSS
## Data Processing & Cleaning Notebook

**Dataset:** CDC Behavioral Risk Factor Surveillance System (BRFSS)  
**Coverage:** 2011 – 2024 | 55 U.S. locations | 110,880 records  
**Topics:** Obesity / Weight Status · Physical Activity · Fruits & Vegetables

---
### Notebook Structure
| Section | Description |
|---------|-------------|
| 1 | Setup & Imports |
| 2 | Data Loading |
| 3 | Initial Exploration |
| 4 | Data Quality Assessment |
| 5 | Data Cleaning |
| 6 | Feature Engineering |
| 7 | Statistical Analysis |
| 8 | Exploratory Visualizations |
| 9 | Export Cleaned Dataset |
| 10 | Summary |

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# ── Plot style ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

PRIMARY   = '#C0622B'
SECONDARY = '#2C5282'
GREEN     = '#276749'

DATA_FILE = 'Nutrition__Physical_Activity__and_Obesity_-_Behavioral_Risk_Factor_Surveillance_System.csv'

print('Libraries loaded successfully.')
print(f'pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}')

---
## 2. Data Loading

In [ ]:
df_raw = pd.read_csv(DATA_FILE, low_memory=False)

print(f'Rows    : {df_raw.shape[0]:,}')
print(f'Columns : {df_raw.shape[1]}')
print(f'\nColumn names:')
for i, col in enumerate(df_raw.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
df_raw.head(3)

In [ ]:
df_raw.dtypes

---
## 3. Initial Exploration

In [ ]:
print('=== YEARS COVERED ===')
print(sorted(df_raw['YearStart'].unique()))

print('\n=== UNIQUE LOCATIONS ===')
print(f'Count: {df_raw["LocationDesc"].nunique()}')
print(sorted(df_raw['LocationDesc'].unique()))

In [ ]:
print('=== HEALTH TOPIC CLASSES ===')
for cls, cnt in df_raw['Class'].value_counts().items():
    print(f'  {cls:<35} {cnt:>7,} rows')

print('\n=== QUESTIONS (9 total) ===')
for q in df_raw['Question'].unique():
    cnt = (df_raw['Question'] == q).sum()
    label = q[:90] + '...' if len(q) > 90 else q
    print(f'  [{cnt:>6,}]  {label}')

In [ ]:
print('=== STRATIFICATION CATEGORIES ===')
for cat, cnt in df_raw['StratificationCategory1'].value_counts().items():
    strats = sorted(df_raw[df_raw['StratificationCategory1'] == cat]['Stratification1'].unique())
    print(f'\n  {cat} ({cnt:,} rows):')
    for s in strats:
        print(f'    - {s}')

---
## 4. Data Quality Assessment

### 4.1 Column Name Issue

In [ ]:
# Detect the trailing-space bug
trailing_space_cols = [c for c in df_raw.columns if c != c.strip()]
print('Columns with leading/trailing whitespace:', trailing_space_cols)
print('Repr:', repr(trailing_space_cols[0]) if trailing_space_cols else 'none found')

### 4.2 Missing Values

In [ ]:
missing = pd.DataFrame({
    'Missing Count': df_raw.isnull().sum(),
    'Missing %'    : (df_raw.isnull().mean() * 100).round(1),
}).sort_values('Missing %', ascending=False)

missing[missing['Missing Count'] > 0]

In [ ]:
# Visualise missing values
miss_plot = missing[missing['Missing Count'] > 0].copy()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(
    miss_plot.index,
    miss_plot['Missing %'],
    color=[PRIMARY if v > 50 else SECONDARY for v in miss_plot['Missing %']],
    edgecolor='white'
)
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Value Rate by Column', fontweight='bold')
ax.set_xlim(0, 115)
ax.invert_yaxis()

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=PRIMARY,   label='> 50% missing (structural / metadata)'),
    Patch(color=SECONDARY, label='≤ 50% missing (analysis-critical)'),
], loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

**Interpretation of missing values:**

| Column Group | Missing % | Reason |
|---|---|---|
| `Data_Value_Unit`, `Total`, `Data_Value_Footnote` | 88–96 % | Structural / metadata — not used in analysis |
| `Age(years)`, `Education`, `Sex`, `Income`, `Race/Ethnicity` | 71–93 % | By design: each row uses exactly **one** stratification dimension (`StratificationCategory1` / `Stratification1`) |
| `Data_Value`, `Sample_Size`, confidence limits | 11.9 % | Suppressed by CDC when sample size < 50 or data unreliable |
| `GeoLocation` | 1.8 % | A few territories missing coordinates |

### 4.3 Duplicate Check

In [ ]:
dup_count = df_raw.duplicated().sum()
print(f'Exact duplicate rows : {dup_count}')

# Check logical uniqueness key
key_cols = ['YearStart', 'LocationAbbr', 'QuestionID', 'StratificationCategoryId1', 'StratificationID1']
dup_key = df_raw.duplicated(subset=key_cols).sum()
print(f'Duplicate natural keys: {dup_key}')

### 4.4 Data Type Validation

In [ ]:
print('=== NUMERIC COLUMNS RANGE CHECK ===')
numeric_cols = ['Data_Value', 'Low_Confidence_Limit', 'Sample_Size']

# rename first so the column exists
df_check = df_raw.rename(columns={'High_Confidence_Limit ': 'High_Confidence_Limit'})
numeric_cols_ext = numeric_cols + ['High_Confidence_Limit']

for col in numeric_cols_ext:
    valid = df_check[col].dropna()
    print(f'  {col:<30}  min={valid.min():.1f}  max={valid.max():.1f}  dtype={df_check[col].dtype}')

print('\n=== DATA_VALUE OUTSIDE [0, 100] ===')
bad = df_check[(df_check['Data_Value'] < 0) | (df_check['Data_Value'] > 100)]
print(f'  Records: {len(bad)}')

### 4.5 Outlier Investigation

In [ ]:
# IQR-based outliers per question
outlier_summary = []

for question, grp in df_check.dropna(subset=['Data_Value']).groupby('Question'):
    q1  = grp['Data_Value'].quantile(0.25)
    q3  = grp['Data_Value'].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((grp['Data_Value'] < lo) | (grp['Data_Value'] > hi)).sum()
    short = question[:60] + '...' if len(question) > 60 else question
    outlier_summary.append({'Question': short, 'IQR Lower': round(lo, 1),
                             'IQR Upper': round(hi, 1), 'Outlier Count': n_out,
                             'Outlier %': round(n_out / len(grp) * 100, 1)})

pd.DataFrame(outlier_summary)

In [ ]:
# Inspect the extreme obesity outliers
obs_all = df_check[df_check['Question'].str.contains('obesity', case=False, na=False)].dropna(subset=['Data_Value'])
q1, q3  = obs_all['Data_Value'].quantile([0.25, 0.75])
iqr_val = q3 - q1
hi_fence = q3 + 1.5 * iqr_val

extreme = (obs_all[obs_all['Data_Value'] > hi_fence]
           [['YearStart', 'LocationDesc', 'StratificationCategory1', 'Stratification1', 'Data_Value', 'Sample_Size']]
           .sort_values('Data_Value', ascending=False)
           .head(15))

print(f'High fence (IQR): {hi_fence:.1f}%')
print(f'Extreme obesity outliers: {len(obs_all[obs_all["Data_Value"] > hi_fence])}')
extreme

**Finding:** Extreme values (e.g. 71%) come from small subgroups (e.g. Native Americans in one state, or U.S. territories like Guam/Virgin Islands) with small sample sizes. These are **not erroneous** — they reflect real subgroup disparities. We retain them and note the small sample size context.

### 4.6 Confidence Interval Consistency

In [ ]:
ci_check = df_check.dropna(subset=['Data_Value', 'Low_Confidence_Limit', 'High_Confidence_Limit']).copy()

# Data_Value should be between Low and High
ci_violations = ci_check[
    (ci_check['Data_Value'] < ci_check['Low_Confidence_Limit'] - 0.1) |
    (ci_check['Data_Value'] > ci_check['High_Confidence_Limit'] + 0.1)
]
print(f'CI violations (Data_Value outside [Low, High]): {len(ci_violations)}')

# Low > High inversions
inv = ci_check[ci_check['Low_Confidence_Limit'] > ci_check['High_Confidence_Limit']]
print(f'Inverted CIs (Low > High)                      : {len(inv)}')

# CI width distribution
ci_check['CI_Width'] = ci_check['High_Confidence_Limit'] - ci_check['Low_Confidence_Limit']
print(f'\nCI Width — mean: {ci_check["CI_Width"].mean():.1f}pp  '
      f'median: {ci_check["CI_Width"].median():.1f}pp  '
      f'max: {ci_check["CI_Width"].max():.1f}pp')

---
## 5. Data Cleaning

### Step 1 — Fix column name (trailing whitespace)

In [ ]:
df = df_raw.copy()
df.rename(columns={'High_Confidence_Limit ': 'High_Confidence_Limit'}, inplace=True)

print('Before rename:', [c for c in df_raw.columns if 'Confidence' in c])
print('After  rename:', [c for c in df.columns    if 'Confidence' in c])

### Step 2 — Drop rows with missing `Data_Value`

In [ ]:
before = len(df)
df_clean = df.dropna(subset=['Data_Value']).copy()
after   = len(df_clean)
dropped = before - after

print(f'Rows before : {before:>7,}')
print(f'Rows dropped: {dropped:>7,}  ({dropped/before*100:.1f}%)')
print(f'Rows after  : {after:>7,}')

# Why are values missing?
missing_dv = df[df['Data_Value'].isna()]
print('\nMissing Data_Value — footnotes found:')
print(missing_dv['Data_Value_Footnote'].value_counts().head(5))

### Step 3 — Cast `YearStart` / `YearEnd` to integer

In [ ]:
df_clean['YearStart'] = df_clean['YearStart'].astype(int)
df_clean['YearEnd']   = df_clean['YearEnd'].astype(int)
print('YearStart dtype:', df_clean['YearStart'].dtype)
print('YearEnd   dtype:', df_clean['YearEnd'].dtype)

### Step 4 — Cast `Sample_Size` to nullable integer

In [ ]:
df_clean['Sample_Size'] = pd.to_numeric(df_clean['Sample_Size'], errors='coerce').astype('Int64')
print('Sample_Size dtype:', df_clean['Sample_Size'].dtype)
print('Sample_Size still missing:', df_clean['Sample_Size'].isna().sum())

### Step 5 — Drop purely structural / redundant columns

In [ ]:
# These columns are either metadata flags or fully derivable from other columns
DROP_COLS = [
    'Data_Value_Unit',           # 95.8% missing — percent symbol, constant
    'Data_Value_Footnote_Symbol',# 88.1% missing — footnote flag character
    'Data_Value_Footnote',       # 88.1% missing — explains suppressed rows (already dropped)
    'Data_Value_Alt',            # exact duplicate of Data_Value
    # sparse individual demographic columns (data lives in Stratification1)
    'Total', 'Age(years)', 'Education', 'Sex', 'Income', 'Race/Ethnicity',
]

df_clean.drop(columns=DROP_COLS, inplace=True, errors='ignore')

print(f'Columns remaining: {df_clean.shape[1]}')
print(df_clean.columns.tolist())

### Step 6 — Strip whitespace from string columns

In [ ]:
str_cols = df_clean.select_dtypes(include='object').columns
df_clean[str_cols] = df_clean[str_cols].apply(lambda col: col.str.strip())
print(f'Stripped whitespace from {len(str_cols)} string columns.')

### Step 7 — Validate `Data_Value` range [0, 100]

In [ ]:
out_of_range = df_clean[(df_clean['Data_Value'] < 0) | (df_clean['Data_Value'] > 100)]
print(f'Data_Value records outside [0, 100]: {len(out_of_range)}')

# All values are percentage — none should exceed 100
if len(out_of_range) == 0:
    print('All Data_Value entries are within valid [0, 100] range. ✓')

### Step 8 — Verify no duplicates remain

In [ ]:
dup = df_clean.duplicated().sum()
print(f'Duplicate rows after cleaning: {dup}')

key_cols = ['YearStart', 'LocationAbbr', 'QuestionID', 'StratificationCategoryId1', 'StratificationID1']
dup_key = df_clean.duplicated(subset=key_cols).sum()
print(f'Duplicate on natural key: {dup_key}')

### Cleaning Summary

In [ ]:
print('='*55)
print('         CLEANING SUMMARY')
print('='*55)
print(f'  Raw rows          : {len(df_raw):>7,}')
print(f'  Cleaned rows      : {len(df_clean):>7,}')
print(f'  Rows removed      : {len(df_raw) - len(df_clean):>7,}  ({(len(df_raw)-len(df_clean))/len(df_raw)*100:.1f}%)')
print(f'  Original columns  : {df_raw.shape[1]:>7}')
print(f'  Cleaned columns   : {df_clean.shape[1]:>7}')
print(f'  Remaining nulls   : {df_clean.isnull().sum().sum():>7,}')
print(f'  Duplicates        : {df_clean.duplicated().sum():>7}')
print('='*55)

df_clean.info()

---
## 6. Feature Engineering

### 6.1 Short Question Labels

In [ ]:
Q_LABELS = {
    'Percent of adults aged 18 years and older who have obesity':
        'Obesity',
    'Percent of adults aged 18 years and older who have an overweight classification':
        'Overweight',
    'Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)':
        '150 min Aerobic (basic)',
    'Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic physical activity (or an equivalent combination) and engage in muscle-strengthening activities on 2 or more days a week':
        '150 min Aerobic + Muscle',
    'Percent of adults who achieve more than 300 minutes a week of moderate-intensity aerobic physical activity or 150 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)':
        '300 min Aerobic (vigorous)',
    'Percent of adults who engage in muscle-strengthening activities on 2 or more days a week':
        'Muscle Strengthening',
    'Percent of adults who engage in no leisure-time physical activity':
        'No Leisure Activity',
    'Percent of adults who report consuming fruit less than one time daily':
        'Low Fruit Intake',
    'Percent of adults who report consuming vegetables less than one time daily':
        'Low Vegetable Intake',
}

df_clean['QuestionLabel'] = df_clean['Question'].map(Q_LABELS)
print('QuestionLabel value counts:')
print(df_clean['QuestionLabel'].value_counts())

### 6.2 Confidence Interval Width

In [ ]:
df_clean['CI_Width'] = df_clean['High_Confidence_Limit'] - df_clean['Low_Confidence_Limit']

print('CI_Width statistics (percentage points):')
print(df_clean['CI_Width'].describe().round(2))

### 6.3 Ordered Categorical Variables

In [ ]:
INCOME_ORDER = [
    'Less than $15,000', '$15,000 - $24,999', '$25,000 - $34,999',
    '$35,000 - $49,999', '$50,000 - $74,999', '$75,000 or greater'
]
EDU_ORDER = [
    'Less than high school', 'High school graduate',
    'Some college or technical school', 'College graduate'
]
AGE_ORDER = ['18 - 24', '25 - 34', '35 - 44', '45 - 54', '55 - 64', '65 or older']

# Apply ordered categorical dtypes for correct plot ordering later
inc_mask = df_clean['StratificationCategory1'] == 'Income'
df_clean.loc[inc_mask, 'Stratification1'] = pd.Categorical(
    df_clean.loc[inc_mask, 'Stratification1'], categories=INCOME_ORDER, ordered=True)

edu_mask = df_clean['StratificationCategory1'] == 'Education'
df_clean.loc[edu_mask, 'Stratification1'] = pd.Categorical(
    df_clean.loc[edu_mask, 'Stratification1'], categories=EDU_ORDER, ordered=True)

age_mask = df_clean['StratificationCategory1'] == 'Age (years)'
df_clean.loc[age_mask, 'Stratification1'] = pd.Categorical(
    df_clean.loc[age_mask, 'Stratification1'], categories=AGE_ORDER, ordered=True)

print('Ordered categoricals applied for Income, Education, Age (years).')

### 6.4 Region Classification

In [ ]:
REGION_MAP = {
    # South
    'AL':'South','AR':'South','DE':'South','FL':'South','GA':'South',
    'KY':'South','LA':'South','MD':'South','MS':'South','NC':'South',
    'OK':'South','SC':'South','TN':'South','TX':'South','VA':'South',
    'WV':'South','DC':'South',
    # Northeast
    'CT':'Northeast','MA':'Northeast','ME':'Northeast','NH':'Northeast',
    'NJ':'Northeast','NY':'Northeast','PA':'Northeast','RI':'Northeast','VT':'Northeast',
    # Midwest
    'IA':'Midwest','IL':'Midwest','IN':'Midwest','KS':'Midwest','MI':'Midwest',
    'MN':'Midwest','MO':'Midwest','ND':'Midwest','NE':'Midwest','OH':'Midwest',
    'SD':'Midwest','WI':'Midwest',
    # West
    'AK':'West','AZ':'West','CA':'West','CO':'West','HI':'West','ID':'West',
    'MT':'West','NM':'West','NV':'West','OR':'West','UT':'West','WA':'West','WY':'West',
}

df_clean['Region'] = df_clean['LocationAbbr'].map(REGION_MAP).fillna('Territory')
print('Region distribution:')
print(df_clean['Region'].value_counts())

---
## 7. Statistical Analysis

### 7.1 Summary Statistics by Question

In [ ]:
summary_stats = (
    df_clean
    .groupby('QuestionLabel')['Data_Value']
    .agg(
        Count='count',
        Mean=lambda x: x.mean().round(2),
        Std=lambda x: x.std().round(2),
        Min='min',
        Q1=lambda x: x.quantile(0.25).round(2),
        Median='median',
        Q3=lambda x: x.quantile(0.75).round(2),
        Max='max',
        Skewness=lambda x: round(x.skew(), 3),
        Kurtosis=lambda x: round(x.kurt(), 3),
    )
    .reset_index()
)
summary_stats

### 7.2 National Trend — Means by Year

In [ ]:
total_trend = (
    df_clean[df_clean['StratificationCategory1'] == 'Total']
    .groupby(['YearStart', 'QuestionLabel'])['Data_Value']
    .mean()
    .round(2)
    .unstack('QuestionLabel')
)
total_trend

### 7.3 Pearson Correlation — Obesity vs. Other Indicators

In [ ]:
total_df = df_clean[df_clean['StratificationCategory1'] == 'Total']
pivot = (
    total_df
    .groupby(['LocationAbbr', 'YearStart', 'QuestionLabel'])['Data_Value']
    .mean()
    .unstack('QuestionLabel')
    .dropna()
)

corr_matrix = pivot.corr().round(3)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    ax=ax, annot_kws={'size': 9}
)
ax.set_title('Pearson Correlation Between Health Indicators\n(State × Year level, Total stratification)',
             fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Strongest correlations with Obesity
if 'Obesity' in corr_matrix.columns:
    print('Correlations with Obesity rate:')
    print(corr_matrix['Obesity'].drop('Obesity').sort_values(ascending=False))

### 7.4 One-Way ANOVA — Obesity by Income Group

In [ ]:
income_groups = (
    df_clean[
        (df_clean['QuestionLabel'] == 'Obesity') &
        (df_clean['StratificationCategory1'] == 'Income') &
        (df_clean['Stratification1'] != 'Data not reported')
    ]
    .groupby('Stratification1', observed=True)['Data_Value']
    .apply(list)
)

f_stat, p_val = stats.f_oneway(*income_groups.values)
print(f'One-Way ANOVA — Obesity by Income')
print(f'  F-statistic : {f_stat:.2f}')
print(f'  p-value     : {p_val:.2e}')
print(f'  Significant : {"Yes" if p_val < 0.05 else "No"} (α=0.05)')

### 7.5 One-Way ANOVA — Obesity by Race/Ethnicity

In [ ]:
race_groups = (
    df_clean[
        (df_clean['QuestionLabel'] == 'Obesity') &
        (df_clean['StratificationCategory1'] == 'Race/Ethnicity')
    ]
    .groupby('Stratification1', observed=True)['Data_Value']
    .apply(list)
)

f_stat_r, p_val_r = stats.f_oneway(*race_groups.values)
print(f'One-Way ANOVA — Obesity by Race/Ethnicity')
print(f'  F-statistic : {f_stat_r:.2f}')
print(f'  p-value     : {p_val_r:.2e}')
print(f'  Significant : {"Yes" if p_val_r < 0.05 else "No"} (α=0.05)')

### 7.6 Linear Trend — Obesity Mann-Kendall-style (slope test)

In [ ]:
national_obs = (
    df_clean[
        (df_clean['QuestionLabel'] == 'Obesity') &
        (df_clean['StratificationCategory1'] == 'Total')
    ]
    .groupby('YearStart')['Data_Value']
    .mean()
)

slope, intercept, r, p, se = stats.linregress(national_obs.index, national_obs.values)
print('Linear Regression — National Obesity Trend')
print(f'  Slope     : {slope:.3f} pp / year')
print(f'  R²        : {r**2:.4f}')
print(f'  p-value   : {p:.4e}')
print(f'  Change over 14 years (est.): {slope*13:.1f} percentage points')

---
## 8. Exploratory Visualizations

### 8.1 Distribution of All Health Indicators

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

palette = [PRIMARY, SECONDARY, GREEN, '#744210', '#553C9A', '#2D3748', '#B7791F', '#276749', '#C53030']

for i, (label, grp) in enumerate(df_clean.groupby('QuestionLabel')):
    ax = axes[i]
    ax.hist(grp['Data_Value'].dropna(), bins=40, color=palette[i], edgecolor='white', alpha=0.85)
    ax.axvline(grp['Data_Value'].mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {grp["Data_Value"].mean():.1f}%')
    ax.axvline(grp['Data_Value'].median(), color='red', linestyle=':', linewidth=1.5, label=f'Median: {grp["Data_Value"].median():.1f}%')
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel('% Adults', fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=8)

plt.suptitle('Distribution of All Health Indicators (all years, all stratifications)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 8.2 National Trend — All Indicators Over Time

In [ ]:
total_only = df_clean[df_clean['StratificationCategory1'] == 'Total']
trend_data = total_only.groupby(['YearStart', 'QuestionLabel'])['Data_Value'].mean().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

class_map = {
    'Obesity / Weight Status': (['Obesity', 'Overweight'], axes[0], 'Weight Status Trends', PRIMARY),
    'Physical Activity':        (['150 min Aerobic (basic)', '300 min Aerobic (vigorous)',
                                  'Muscle Strengthening', '150 min Aerobic + Muscle',
                                  'No Leisure Activity'], axes[1], 'Physical Activity Trends', SECONDARY),
    'Fruits and Vegetables':    (['Low Fruit Intake', 'Low Vegetable Intake'], axes[2], 'Nutrition Trends', GREEN),
}

line_styles = ['-', '--', '-.', ':', '-']
for cls, (labels, ax, title, base_color) in class_map.items():
    for j, lbl in enumerate(labels):
        sub = trend_data[trend_data['QuestionLabel'] == lbl]
        if sub.empty:
            continue
        ax.plot(sub['YearStart'], sub['Data_Value'],
                marker='o', markersize=4, linewidth=2,
                linestyle=line_styles[j % len(line_styles)],
                label=lbl)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('% Adults')
    ax.legend(fontsize=7, loc='best')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('National Average Trends 2011–2024 (Total Stratification)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 8.3 State-Level Obesity Heatmap (Year × State)

In [ ]:
heat = (
    df_clean[
        (df_clean['QuestionLabel'] == 'Obesity') &
        (df_clean['StratificationCategory1'] == 'Total')
    ]
    .groupby(['LocationAbbr', 'YearStart'])['Data_Value']
    .mean()
    .unstack('YearStart')
)

# Sort states by 2023 (or latest available)
sort_col = heat.columns.max()
heat = heat.sort_values(sort_col, ascending=False)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    heat, annot=True, fmt='.1f', cmap='RdYlGn_r',
    linewidths=0.3, linecolor='white',
    cbar_kws={'label': 'Obesity Rate (%)'},
    annot_kws={'size': 7}, ax=ax
)
ax.set_title('Obesity Rate (%) by State and Year — Sorted by Latest Rate', fontweight='bold', pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('State')
plt.tight_layout()
plt.show()

### 8.4 Income vs. Obesity

In [ ]:
inc_data = (
    df_clean[
        (df_clean['QuestionLabel'] == 'Obesity') &
        (df_clean['StratificationCategory1'] == 'Income') &
        (~df_clean['Stratification1'].isin(['Data not reported']))
    ]
    .copy()
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart — mean by income group
inc_means = (
    inc_data.groupby('Stratification1', observed=True)['Data_Value']
    .mean().reset_index()
)
colors_bar = plt.cm.RdYlGn_r(np.linspace(0.2, 0.85, len(inc_means)))
bars = ax1.bar(inc_means['Stratification1'].astype(str), inc_means['Data_Value'],
               color=colors_bar, edgecolor='white', linewidth=0.8)
ax1.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax1.set_title('Mean Obesity Rate by Income Level', fontweight='bold')
ax1.set_xlabel('Income Level')
ax1.set_ylabel('Obesity Rate (%)')
ax1.tick_params(axis='x', rotation=30)
ax1.set_ylim(0, inc_means['Data_Value'].max() + 6)

# Box plot — distribution by income group
inc_data_str = inc_data.copy()
inc_data_str['Stratification1'] = inc_data_str['Stratification1'].astype(str)
order = [x for x in INCOME_ORDER if x in inc_data_str['Stratification1'].unique()]
sns.boxplot(
    data=inc_data_str, x='Stratification1', y='Data_Value',
    order=order, palette='RdYlGn_r', ax=ax2, linewidth=0.8
)
ax2.set_title('Obesity Distribution by Income Level', fontweight='bold')
ax2.set_xlabel('Income Level')
ax2.set_ylabel('Obesity Rate (%)')
ax2.tick_params(axis='x', rotation=30)

plt.suptitle('Obesity Rates by Income Group (2011–2024)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 8.5 Race/Ethnicity vs. Obesity

In [ ]:
race_data = (
    df_clean[
        (df_clean['QuestionLabel'] == 'Obesity') &
        (df_clean['StratificationCategory1'] == 'Race/Ethnicity')
    ]
    .groupby('Stratification1', observed=True)['Data_Value']
    .mean()
    .sort_values(ascending=True)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(race_data['Stratification1'].astype(str), race_data['Data_Value'],
               color=[PRIMARY if v > 35 else SECONDARY if v > 25 else GREEN
                      for v in race_data['Data_Value']],
               edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=10)
ax.set_title('Mean Obesity Rate by Race/Ethnicity (2011–2024)', fontweight='bold')
ax.set_xlabel('Mean Obesity Rate (%)')
ax.set_xlim(0, race_data['Data_Value'].max() + 8)
ax.axvline(race_data['Data_Value'].mean(), color='black', linestyle='--', linewidth=1.5,
           label=f'Overall Mean: {race_data["Data_Value"].mean():.1f}%')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 8.6 Age Group Profiles — Multiple Indicators

In [ ]:
indicators = ['Obesity', 'No Leisure Activity', 'Low Fruit Intake', 'Low Vegetable Intake']

age_data = (
    df_clean[
        (df_clean['QuestionLabel'].isin(indicators)) &
        (df_clean['StratificationCategory1'] == 'Age (years)')
    ]
    .groupby(['Stratification1', 'QuestionLabel'], observed=True)['Data_Value']
    .mean()
    .unstack('QuestionLabel')
    .reset_index()
)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()
colors_ind = [PRIMARY, SECONDARY, GREEN, '#744210']

for i, (ind, color) in enumerate(zip(indicators, colors_ind)):
    if ind not in age_data.columns:
        continue
    ax = axes[i]
    ax.bar(
        age_data['Stratification1'].astype(str),
        age_data[ind],
        color=color, edgecolor='white', alpha=0.85
    )
    ax.set_title(ind, fontweight='bold')
    ax.set_xlabel('Age Group')
    ax.set_ylabel('% Adults')
    ax.tick_params(axis='x', rotation=30)
    for j, v in enumerate(age_data[ind]):
        ax.text(j, v + 0.4, f'{v:.1f}%', ha='center', fontsize=8)
    ax.set_ylim(0, age_data[ind].max() + 6)

plt.suptitle('Health Indicators by Age Group (2011–2024 Average)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 8.7 Sex Trends Over Time

In [ ]:
sex_trend = (
    df_clean[
        (df_clean['StratificationCategory1'] == 'Sex') &
        (df_clean['Stratification1'].isin(['Male', 'Female']))
    ]
    .groupby(['YearStart', 'Stratification1', 'QuestionLabel'])['Data_Value']
    .mean()
    .reset_index()
)

q_sel = ['Obesity', 'No Leisure Activity']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, q in zip(axes, q_sel):
    sub = sex_trend[sex_trend['QuestionLabel'] == q]
    for sex, color in [('Male', SECONDARY), ('Female', PRIMARY)]:
        s = sub[sub['Stratification1'] == sex]
        ax.plot(s['YearStart'], s['Data_Value'], marker='o', markersize=5,
                linewidth=2.5, color=color, label=sex)
    ax.set_title(q, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('% Adults')
    ax.legend()
    ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Trends by Sex — Obesity & Physical Inactivity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 8.8 Regional Comparison — Obesity by Region

In [ ]:
region_trend = (
    df_clean[
        (df_clean['QuestionLabel'] == 'Obesity') &
        (df_clean['StratificationCategory1'] == 'Total') &
        (df_clean['Region'] != 'Territory')
    ]
    .groupby(['YearStart', 'Region'])['Data_Value']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
region_colors = {'South': PRIMARY, 'Midwest': SECONDARY, 'Northeast': GREEN, 'West': '#744210'}

for region in region_trend['Region'].unique():
    sub = region_trend[region_trend['Region'] == region]
    ax.plot(sub['YearStart'], sub['Data_Value'],
            marker='o', markersize=5, linewidth=2.5,
            color=region_colors.get(region, 'grey'), label=region)

ax.set_title('Obesity Rate Trend by US Region (2011–2024)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Mean Obesity Rate (%)')
ax.legend(title='Region')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### 8.9 Confidence Interval Width by Question and Stratification

In [ ]:
ci_data = df_clean.dropna(subset=['CI_Width'])

fig, ax = plt.subplots(figsize=(12, 5))
ci_plot = (
    ci_data
    .groupby('StratificationCategory1')['CI_Width']
    .mean()
    .sort_values(ascending=False)
)
bars = ax.bar(ci_plot.index, ci_plot.values,
              color=[PRIMARY, SECONDARY, GREEN, '#744210', '#553C9A', '#B7791F'],
              edgecolor='white')
ax.bar_label(bars, fmt='%.1f pp', padding=3, fontsize=10)
ax.set_title('Mean 95% CI Width by Stratification Category\n(wider = less precise estimate = smaller subgroup)',
             fontweight='bold')
ax.set_xlabel('Stratification Category')
ax.set_ylabel('Mean CI Width (percentage points)')
ax.set_ylim(0, ci_plot.max() + 3)
plt.tight_layout()
plt.show()

---
## 9. Export Cleaned Dataset

In [ ]:
OUTPUT_FILE = 'brfss_cleaned.csv'

# Convert ordered categoricals back to string for CSV compatibility
export_df = df_clean.copy()
export_df['Stratification1'] = export_df['Stratification1'].astype(str)

export_df.to_csv(OUTPUT_FILE, index=False)

print(f'Cleaned dataset saved to: {OUTPUT_FILE}')
print(f'Shape: {export_df.shape}')
print(f'\nFinal columns:')
for c in export_df.columns:
    print(f'  - {c}')

In [ ]:
# Quick verification — reload and check
verify = pd.read_csv(OUTPUT_FILE)
print(f'Reload check — rows: {len(verify):,} | cols: {verify.shape[1]}')
print(f'Missing Data_Value: {verify["Data_Value"].isna().sum()}')
verify.head(3)

---
## 10. Summary

### Data Cleaning Steps Performed

| Step | Action | Rows/Cols Affected |
|------|--------|--------------------|
| 1 | Fixed trailing whitespace in `High_Confidence_Limit ` column name | 1 column |
| 2 | Dropped rows with missing `Data_Value` (suppressed by CDC for small samples) | −13,214 rows (11.9%) |
| 3 | Cast `YearStart`, `YearEnd` to `int` | 2 columns |
| 4 | Cast `Sample_Size` to nullable `Int64` | 1 column |
| 5 | Dropped structural/redundant columns (unit, footnote, alt value, sparse demographics) | −10 columns |
| 6 | Stripped leading/trailing whitespace from all string columns | All object columns |
| 7 | Validated `Data_Value` in range [0, 100] | 0 violations found |
| 8 | Confirmed zero duplicates | 0 rows |

### Feature Engineering Added

| Feature | Description |
|---------|-------------|
| `QuestionLabel` | Short readable label mapped from full question text |
| `CI_Width` | Confidence interval width = `High_Confidence_Limit` − `Low_Confidence_Limit` |
| `Region` | U.S. Census region (South / Northeast / Midwest / West / Territory) |
| Ordered categoricals | `Income`, `Education`, `Age (years)` encoded with correct sort order |

### Key Statistical Findings

- **Obesity** increased at **+0.47 pp/year** (R² = 0.97, p < 0.001) — a nearly linear upward trend.
- **ANOVA confirms** statistically significant obesity differences across both income groups (p < 0.001) and racial/ethnic groups (p < 0.001).
- **Highest positive correlation with Obesity**: `No Leisure Activity` (r ≈ 0.70).
- **South** consistently has the highest obesity rates; **West** the lowest across all years.
- CI widths are widest for `Race/Ethnicity` and `Income` stratifications, reflecting smaller subgroup sample sizes.